# Google ADK: Build AI Agents — Complete Colab Notebook

> **Three Projects in One** · Groq + LLaMA instead of Gemini

| # | Project | Core Concepts |
|---|---|---|
| 0 | Persistent Storage with ADK | `DatabaseSessionService`, tool-driven CRUD on `ToolContext.state` |
| 1 | ADK API Server + Streamlit Precursor | REST endpoints, session lifecycle, Streamlit client |
| 2 | Capital Agent + Cloud Run | Minimal agent structure · deployment walkthrough (markdown) |

**Prerequisite:** Save your Groq key as a Colab Secret named `GROQ_API_KEY`  
*(Runtime → Secrets → Add new secret)*


---
## ⚙️ Step 0 — Environment Setup

### 0.1 Install dependencies

In [12]:
!pip install -q google-adk groq litellm python-dotenv sqlalchemy nest-asyncio aiosqlite

### 0.2 Load GROQ_API_KEY from Colab Secrets

In [3]:
import os
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
os.environ['GROQ_API_KEY']   = GROQ_API_KEY
# LiteLLM uses OpenAI-compatible interface — point it at Groq
os.environ['OPENAI_API_KEY']  = GROQ_API_KEY
os.environ['OPENAI_API_BASE'] = 'https://api.groq.com/openai/v1'

print('GROQ_API_KEY loaded:', bool(GROQ_API_KEY))


GROQ_API_KEY loaded: True


### 0.3 Verify Groq connectivity

In [4]:
from groq import Groq

_client = Groq(api_key=GROQ_API_KEY)
_resp = _client.chat.completions.create(
    model='llama-3.1-8b-instant',
    messages=[{'role':'user','content':'Reply with exactly: Groq OK'}],
    max_tokens=10,
)
print(_resp.choices[0].message.content)


Groq OK


### 0.4 Model constants

| Constant | Model string | Best for |
|---|---|---|
| `INSTANT_MODEL` | `groq/llama-3.1-8b-instant` | Fast, simple tasks |
| `VERSATILE_MODEL` | `groq/llama-3.3-70b-versatile` | Reasoning, tool use |


In [5]:
INSTANT_MODEL   = 'groq/llama-3.1-8b-instant'
VERSATILE_MODEL = 'groq/llama-3.3-70b-versatile'
DEFAULT_MODEL   = VERSATILE_MODEL
print('Default LLM:', DEFAULT_MODEL)


Default LLM: groq/llama-3.3-70b-versatile


---
# 📚 Project 0 — Persistent Storage with ADK
### Reading List Curator

**What you learn:**
- `InMemorySessionService` vs `DatabaseSessionService` (SQLite)
- Tools that read/write `tool_context.state` → auto-persisted by session service
- `Runner.run_async()` event streaming

**Folder structure (mirrors course):**
```
project_0/
├── memory_agent/
│   ├── __init__.py
│   └── agent.py      ← LlmAgent + 6 CRUD tools
├── utils.py          ← event logger + state printer
└── reading_list.db   ← auto-created SQLite file
```


### Step 1 — Create folder structure

In [6]:
import os
os.makedirs('project_0/memory_agent', exist_ok=True)
print('Folders created: project_0/memory_agent/')


Folders created: project_0/memory_agent/


### Step 2 — `memory_agent/__init__.py`

In [7]:
%%writefile project_0/memory_agent/__init__.py
# Package marker — required for ADK auto-discovery


Writing project_0/memory_agent/__init__.py


### Step 3 — `memory_agent/agent.py`

Six tools that operate on `tool_context.state`:

| Tool | What it does |
|---|---|
| `set_user_name` | Persist user display name |
| `add_item` | Append `{title, url, tags, status, notes}` |
| `list_items` | Return items (optional filter by status/tag) |
| `update_item` | Edit fields by 1-based index |
| `annotate_item` | Replace notes for an item |
| `remove_item` | Delete item by 1-based index |


In [17]:
%%writefile project_0/memory_agent/agent.py
"""
Reading List Curator Agent — adapted for Groq/LLaMA via LiteLLM.
Original: gemini-2.0-flash  |  Adapted: groq/llama-3.3-70b-versatile
"""
import os
from typing import List, Optional
from google.adk.agents import LlmAgent
from google.adk.tools.tool_context import ToolContext

MODEL = os.getenv("ADK_MODEL", "groq/llama-3.3-70b-versatile")

# ── Helpers ──────────────────────────────────────────────────────────

def _ensure_state(tc: ToolContext) -> None:
    """Guarantee required keys exist in state."""
    if "user_name" not in tc.state or tc.state["user_name"] is None:
        tc.state["user_name"] = ""
    if "reading_list" not in tc.state or tc.state["reading_list"] is None:
        tc.state["reading_list"] = []

def _normalize_tags(tags: Optional[List[str]]) -> List[str]:
    if not tags:
        return []
    return [str(t).strip() for t in tags if str(t).strip()]

def _valid_status(status: Optional[str]) -> bool:
    return status in {None, "queued", "reading", "done"}

# ── Tools ─────────────────────────────────────────────────────────────

def set_user_name(name: str, tool_context: ToolContext) -> dict:
    """Set the user's display name in persistent state."""
    _ensure_state(tool_context)
    old = tool_context.state.get("user_name", "")
    tool_context.state["user_name"] = name or ""
    return {
        "action": "set_user_name",
        "old_name": old,
        "new_name": tool_context.state["user_name"],
        "message": f"Saved your name as '{tool_context.state['user_name'] or 'Unknown'}'.",
    }

def add_item(
    title: str,
    url: Optional[str] = None, # Made optional with default None
    tags: Optional[List[str]] = None, # Made optional with default None
    status: Optional[str] = None, # Made optional with default None
    notes: Optional[str] = None,  # Made optional with default None
    tool_context: ToolContext = None,
) -> dict:
    """Add a new entry to the reading list.
    Fields: title (required), url (optional), tags (optional list),
    status (queued|reading|done; default queued), notes (optional).
    """
    _ensure_state(tool_context)
    if not _valid_status(status):
        status = "queued"
    item = {
        "title":  title.strip() if title else "(untitled)",
        "url":    (url or "").strip(),
        "tags":   _normalize_tags(tags),
        "status": status,
        "notes":  (notes or "").strip(),
    }
    rl = tool_context.state["reading_list"]
    rl.append(item)
    tool_context.state["reading_list"] = rl
    return {"action": "add_item", "item": item, "index": len(rl),
            "message": f"Added '{item['title']}' to your reading list."}

def list_items(
    filter_status: Optional[str] = None,
    filter_tag: Optional[str] = None,
    tool_context: ToolContext = None,
) -> dict:
    """Return the reading list, optionally filtered by status or tag."""
    _ensure_state(tool_context)
    rl = tool_context.state["reading_list"]
    filtered = [
        it for it in rl
        if (not filter_status or it.get("status") == filter_status)
        and (not filter_tag or filter_tag in it.get("tags", []))
    ]
    return {"action": "list_items", "count": len(filtered), "items": filtered,
            "filters": {"status": filter_status, "tag": filter_tag},
            "message": f"Found {len(filtered)} item(s)."}

def update_item(
    index: int,
    title: Optional[str] = None,
    url: Optional[str] = None,
    status: Optional[str] = None,
    notes: Optional[str] = None,
    tags: Optional[List[str]] = None,
    tool_context: ToolContext = None,
) -> dict:
    """Update fields of an existing reading-list item (1-based index)."""
    _ensure_state(tool_context)
    rl = tool_context.state["reading_list"]
    if index < 1 or index > len(rl):
        return {"action": "update_item", "status": "error",
                "message": f"No item at position {index}. You have {len(rl)} item(s)."}
    item = rl[index - 1]
    before = item.copy()
    if title  is not None: item["title"]  = title.strip() or item["title"]
    if url    is not None: item["url"]    = (url or "").strip()
    if _valid_status(status) and status is not None: item["status"] = status
    if notes  is not None: item["notes"]  = (notes or "").strip()
    if tags   is not None: item["tags"]   = _normalize_tags(tags)
    rl[index - 1] = item
    tool_context.state["reading_list"] = rl
    return {"action": "update_item", "index": index, "before": before,
            "after": item, "message": f"Updated item {index} ('{before.get('title', '')}')."}

def annotate_item(index: int, notes: str, tool_context: ToolContext) -> dict:
    """Append or set notes for an item (1-based index)."""
    _ensure_state(tool_context)
    rl = tool_context.state["reading_list"]
    if index < 1 or index > len(rl):
        return {"action": "annotate_item", "status": "error",
                "message": f"No item at position {index}. You have {len(rl)} item(s)."}
    item = rl[index - 1]
    before_notes = item.get("notes", "")
    item["notes"] = (notes or "").strip()
    rl[index - 1] = item
    tool_context.state["reading_list"] = rl
    return {"action": "annotate_item", "index": index, "old_notes": before_notes,
            "new_notes": item["notes"], "message": f"Noted item {index} ('{item.get('title', '')}')."}

def remove_item(index: int, tool_context: ToolContext) -> dict:
    """Remove a reading-list item (1-based index)."""
    _ensure_state(tool_context)
    rl = tool_context.state["reading_list"]
    if index < 1 or index > len(rl):
        return {"action": "remove_item", "status": "error",
                "message": f"No item at position {index}. You have {len(rl)} item(s)."}
    removed = rl.pop(index - 1)
    tool_context.state["reading_list"] = rl
    return {"action": "remove_item", "index": index, "removed": removed,
            "message": f"Removed '{removed.get('title', '')}' from your reading list."}

# ── Agent ─────────────────────────────────────────────────────────────

reading_agent = LlmAgent(
    name="reading_list_curator",
    model=MODEL,
    description="Curate a personal reading list with persistent memory.",
    instruction="""
You are a friendly Reading List Curator. The session state contains:
  - user_name: the user's display name (string, may be empty)
  - reading_list: an array of items, each with {title, url, tags[], status, notes}

Your job:
  1) Greet the user by name if known.
  2) Understand natural-language requests and call the appropriate tools.
  3) Return a short, helpful summary after tool calls.

Tool selection:
  - "add" requests   → add_item  (title required; url/tags/status/notes optional; default status=queued)
  - "show/list"      → list_items (pass filter_status or filter_tag if mentioned)
  - update fields    → update_item (infer 1-based index from phrasing)
  - add/replace notes→ annotate_item
  - delete/remove    → remove_item
  - user shares name → set_user_name

Formatting: numbered list — Title [status], then url/tags/notes on sub-lines if present.
Be concise. Never fabricate URLs or tags.
    """,
    tools=[set_user_name, add_item, list_items, update_item, annotate_item, remove_item],
)

root_agent = reading_agent  # ADK auto-discovery alias


Overwriting project_0/memory_agent/agent.py


In [18]:
%%writefile project_0/utils.py
"""
Utility helpers for the Reading List Curator.
Unchanged from course — ANSI colours work in Colab terminals too.
"""
from google.genai import types


class Colors:
    RESET   = "\033[0m";  BOLD    = "\033[1m"
    BLACK   = "\033[30m"; CYAN    = "\033[36m"; GREEN  = "\033[32m"
    BG_BLUE = "\033[44m"; BG_GREEN= "\033[42m"; BG_RED = "\033[41m"


async def display_state_async(session_service, app_name, user_id, session_id, label="State"):
    """Pretty-print current session state."""
    try:
        session = await session_service.get_session(
            app_name=app_name, user_id=user_id, session_id=session_id
        )
        st = session.state or {}
        print(f"\n{'-'*12} {label} {'-'*12}")
        print(f"User: {st.get('user_name', '') or 'Unknown'}")
        items = st.get("reading_list", [])
        if not items:
            print("Reading List: [empty]")
        else:
            print("Reading List:")
            for i, it in enumerate(items, 1):
                print(f"  {i}. {it.get('title','(untitled)')}  [{it.get('status','queued')}]")
                if it.get("url"):   print(f"     URL: {it['url']}")
                if it.get("tags"):  print(f"     Tags: {', '.join(it['tags'])}")
                if it.get("notes"): print(f"     Notes: {it['notes']}")
        print("-" * (26 + len(label)))
    except Exception as e:
        print(f"Error displaying state: {e}")


async def process_agent_response(event):
    """Stream and log events; return final response text."""
    print(f"Event ID: {event.id}, Author: {event.author}")
    if event.content and event.content.parts:
        for part in event.content.parts:
            if getattr(part, "text", None) and part.text.strip():
                print(f"  Text: '{part.text.strip()}'")
            if getattr(part, "tool_response", None):
                print(f"  Tool Response: {part.tool_response.output}")
    if event.is_final_response():
        final_text = ""
        if event.content and event.content.parts and getattr(event.content.parts[0], "text", None):
            final_text = (event.content.parts[0].text or "").strip()
        if final_text:
            print(f"\n{Colors.BG_BLUE}{Colors.BLACK if hasattr(Colors,'BLACK') else ''}{Colors.BOLD}")
            print(f"AGENT RESPONSE:\n{final_text}")
            print(f"{Colors.RESET}")
        return final_text
    return None


async def call_agent_async(runner, user_id, session_id, query: str):
    """Send a query to the agent and return final response text."""
    content = types.Content(role="user", parts=[types.Part(text=query)])
    print(f"\n{Colors.BG_GREEN}{Colors.BOLD}--- Query: {query} ---{Colors.RESET}")
    final_response_text = None
    try:
        async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
            maybe_text = await process_agent_response(event)
            if maybe_text:
                final_response_text = maybe_text
    except Exception as e:
        print(f"Error during agent call: {e}")
    return final_response_text


Overwriting project_0/utils.py


### Step 5 — Demo A: InMemorySessionService
State lives only for the duration of this Python process — good for quick tests.


In [19]:
import asyncio, sys, os, nest_asyncio
nest_asyncio.apply()   # allow asyncio.run() inside Colab event loop

sys.path.insert(0, 'project_0')
os.environ['ADK_MODEL'] = DEFAULT_MODEL

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from memory_agent.agent import root_agent
from utils import call_agent_async, display_state_async

APP_NAME   = 'reading_list_demo'
USER_ID    = 'colab_user'
SESSION_ID = 'demo_session_01'
INITIAL_STATE = {'user_name': '', 'reading_list': []}

in_memory_service = InMemorySessionService()

async def run_in_memory_demo():
    session = await in_memory_service.create_session(
        app_name=APP_NAME, user_id=USER_ID,
        session_id=SESSION_ID, state=INITIAL_STATE
    )
    print(f'Session created: {session.id}')
    runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=in_memory_service)

    queries = [
        'My name is Mayank',
        'Add "Clean Code" with tag software',
        'Add "Attention Is All You Need" url https://arxiv.org/abs/1706.03762 tag nlp',
        'List all items',
    ]
    for q in queries:
        await display_state_async(in_memory_service, APP_NAME, USER_ID, SESSION_ID, 'BEFORE')
        await call_agent_async(runner, USER_ID, SESSION_ID, q)
        await display_state_async(in_memory_service, APP_NAME, USER_ID, SESSION_ID, 'AFTER')

asyncio.run(run_in_memory_demo())


Session created: demo_session_01

------------ BEFORE ------------
User: Unknown
Reading List: [empty]
--------------------------------

--- Query: My name is Mayank ---

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Error during agent call: litellm.BadRequestError: GroqException - {"error":{"message":"Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.","type":"invalid_request_error","code":"tool_use_failed","failed_generation":"\u003cfunction=set_user_name{\"name\": \"Mayank\"}\u003c/function\u003e"}}


------------ AFTER ------------
User: Unknown
Reading List: [empty]
-------------------------------

------------ BEFORE ------------
User: Unknown
Reading List: [empty]
--------------------------------

--- Query: Add "Clean Code" with tag software ---

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info

### Step 6 — Demo B: DatabaseSessionService (SQLite)
State is written to `project_0/reading_list.db` and survives across `asyncio.run()` calls,  
simulating real cross-run persistence.


In [20]:
from google.adk.sessions import DatabaseSessionService

DB_URL      = 'sqlite+aiosqlite:///./project_0/reading_list.db'
APP_NAME_DB = 'reading_list_persistent'
USER_ID_DB  = 'colab_user_db'

db_service = DatabaseSessionService(db_url=DB_URL)

async def get_or_create_session():
    existing = await db_service.list_sessions(app_name=APP_NAME_DB, user_id=USER_ID_DB)
    if getattr(existing, 'sessions', None):
        sid = existing.sessions[0].id
        print(f'Continuing existing session: {sid}')
    else:
        new_s = await db_service.create_session(
            app_name=APP_NAME_DB, user_id=USER_ID_DB,
            state={'user_name': '', 'reading_list': []}
        )
        sid = new_s.id
        print(f'Created new session: {sid}')
    return sid

SESSION_ID_DB = asyncio.run(get_or_create_session())

Continuing existing session: a8784a74-b6d4-4762-9ee5-497ad511134e


In [22]:
# Run 1 — seed the database
async def run_persistent_demo_1():
    runner = Runner(agent=root_agent, app_name=APP_NAME_DB, session_service=db_service)
    for q in [
        'My name is Colab User',
        'Add "Deep Learning" tag ai',
        'Add "The Pragmatic Programmer" tag software',
        'Show my list',
    ]:
        await call_agent_async(runner, USER_ID_DB, SESSION_ID_DB, q)
    await display_state_async(db_service, APP_NAME_DB, USER_ID_DB, SESSION_ID_DB, 'State after Run 1')

asyncio.run(run_persistent_demo_1())



--- Query: My name is Colab User ---
Event ID: 73272d14-474a-42da-9f58-11c3dbfe1f54, Author: reading_list_curator
Event ID: 9d1d2070-aa05-437d-bed6-faa0567c253f, Author: reading_list_curator

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Error during agent call: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01k8zbvz9xf3v97wzxqpfaxz6g` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11591, Requested 2437. Please try again in 10.14s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


--- Query: Add "Deep Learning" tag ai ---

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debu

In [23]:
# Run 2 — mutate state (simulates new process reading persisted DB)
async def run_persistent_demo_2():
    runner = Runner(agent=root_agent, app_name=APP_NAME_DB, session_service=db_service)
    for q in [
        'Update item 1 status reading',
        'Annotate item 2: must-read for any dev team',
        'Remove item 1',
        'Show all items',
    ]:
        await call_agent_async(runner, USER_ID_DB, SESSION_ID_DB, q)
    await display_state_async(db_service, APP_NAME_DB, USER_ID_DB, SESSION_ID_DB, 'State after Run 2')

asyncio.run(run_persistent_demo_2())



--- Query: Update item 1 status reading ---
Event ID: 37b976b3-b494-45ec-b4e0-96663d44e102, Author: reading_list_curator
Event ID: c65dc9ee-a839-41dc-ba70-338956127eca, Author: reading_list_curator

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Error during agent call: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01k8zbvz9xf3v97wzxqpfaxz6g` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11045, Requested 2574. Please try again in 8.094999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


--- Query: Annotate item 2: must-read for any dev team ---

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this err

### Step 7 — Inspect the SQLite database

In [24]:
import sqlite3
conn = sqlite3.connect('project_0/reading_list.db')
cur  = conn.cursor()
cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
print('Tables:', cur.fetchall())
cur.execute('SELECT id, app_name, user_id FROM sessions LIMIT 5')
for r in cur.fetchall(): print('Session row:', r)
conn.close()


Tables: [('adk_internal_metadata',), ('sessions',), ('app_states',), ('user_states',), ('events',)]
Session row: ('a8784a74-b6d4-4762-9ee5-497ad511134e', 'reading_list_persistent', 'colab_user_db')


---
# 🌐 Project 1 — ADK API Server + Streamlit Precursor
### Simple Q&A Agent

**What you learn:**
- ADK API server endpoints: `POST /run`, `POST /apps/.../sessions/...`, `GET /list-apps`
- Parsing multi-event API responses (`events[]` array)
- `ADKClient` HTTP helper
- Building a Streamlit client (full `app.py` written to disk; curl reference included)

**Folder structure (mirrors course):**
```
project_1/
├── agents/
│   └── simple/
│       ├── __init__.py
│       └── agent.py       ← Q&A LlmAgent (Groq/LLaMA)
├── common/
│   └── adk_client.py      ← minimal HTTP client helper
├── apps/
│   └── app.py             ← Streamlit frontend
└── scripts/               ← shell helpers (reference)
```

> **Colab note:** We cannot run a long-lived `adk api_server` and call it from the same kernel.  
> The Python SDK path (Runner + InMemorySessionService) is used for live demo — identical agent logic, no HTTP layer.


### Step 1 — Create folder structure

In [25]:
for d in ['project_1/agents/simple','project_1/common','project_1/apps','project_1/scripts']:
    os.makedirs(d, exist_ok=True)
print('project_1/ structure ready')


project_1/ structure ready


In [26]:
%%writefile project_1/agents/simple/__init__.py
# Package marker


Writing project_1/agents/simple/__init__.py


In [27]:
%%writefile project_1/agents/simple/agent.py
"""
Simple Q&A agent — adapted for Groq/LLaMA.
Course original used Tavily MCP for search.
In Colab we keep it tool-free to avoid npx/Node.js friction.
Add MCPToolset here if TAVILY_API_KEY is available.
"""
import os
from google.adk.agents.llm_agent import LlmAgent

MODEL = os.getenv("ADK_MODEL", "groq/llama-3.3-70b-versatile")

root_agent = LlmAgent(
    model=MODEL,
    name="simple",
    instruction="""
You are a helpful Q&A assistant. Answer factual questions clearly in 3-6 sentences.
Cite your reasoning. If uncertain, say so honestly.
    """.strip(),
    tools=[],  # attach MCPToolset(tavily) here when TAVILY_API_KEY is set
)


Writing project_1/agents/simple/agent.py


In [28]:
%%writefile project_1/common/adk_client.py
"""
Minimal HTTP client for the ADK API Server (unchanged from course).
Used by Streamlit app and curl-equivalent scripts.
"""
import os, time, requests
from typing import Any, Dict, Optional

API_BASE = os.getenv("ADK_API_BASE", "http://localhost:8000")


class ADKClient:
    """Thin wrapper: create session + run one turn."""

    def __init__(self, api_base: Optional[str] = None):
        self.api_base = api_base or API_BASE

    def create_session(self, app_name: str, user_id: str,
                       session_id: Optional[str] = None) -> str:
        sid = session_id or f"session-{int(time.time())}"
        url = f"{self.api_base}/apps/{app_name}/users/{user_id}/sessions/{sid}"
        requests.post(url, headers={"Content-Type": "application/json"},
                      json={}).raise_for_status()
        return sid

    def run(self, app_name: str, user_id: str, session_id: str,
            message: str) -> Dict[str, Any]:
        payload = {
            "app_name": app_name, "user_id": user_id, "session_id": session_id,
            "new_message": {"role": "user", "parts": [{"text": message}]},
        }
        r = requests.post(f"{self.api_base}/run",
                          headers={"Content-Type": "application/json"},
                          json=payload)
        r.raise_for_status()
        try:    return r.json()
        except: return {"raw": r.text}

    @staticmethod
    def parse_events_for_text(resp: Dict[str, Any]) -> str:
        """Extract final assistant text from ADK event payload."""
        events = resp if isinstance(resp, list) else resp.get("events", [resp])
        final_text = ""
        for ev in events:
            content = ev.get("content", {})
            parts = content.get("parts", []) if isinstance(content, dict) else []
            for p in parts:
                if isinstance(p.get("text"), str) and p["text"].strip():
                    final_text = p["text"].strip()
        return final_text


Writing project_1/common/adk_client.py


In [29]:
%%writefile project_1/apps/app.py
"""
ADK API + Streamlit — Simple Q&A app (unchanged from course).
Run locally after launching: adk api_server -v project_1/
    streamlit run project_1/apps/app.py
"""
from __future__ import annotations
import os, uuid, json, requests
import streamlit as st

API_DEFAULT  = os.getenv("ADK_API_BASE", "http://localhost:8000")
APP_DEFAULT  = os.getenv("ADK_APP_SIMPLE", "simple")
USER_DEFAULT = os.getenv("ADK_USER_ID", f"user-{uuid.uuid4()}")

st.set_page_config(page_title="ADK + Streamlit: Simple Q&A", layout="centered")
st.title("Q&A — ADK API + Streamlit")
st.caption("Frontend: Streamlit  ·  Backend: ADK API  ·  Model: Groq/LLaMA")

for k, v in [("api_base", API_DEFAULT), ("app_name", APP_DEFAULT),
              ("user_id", USER_DEFAULT), ("session_id", None), ("events", [])]:
    if k not in st.session_state:
        st.session_state[k] = v

def GET(url):
    r = requests.get(url, timeout=60); r.raise_for_status(); return r.json()

def POST(url, payload):
    r = requests.post(url, json=payload,
                      headers={"Content-Type": "application/json"}, timeout=120)
    r.raise_for_status()
    try:    return r.json()
    except: return {"raw": r.text}

def create_session():
    sid = f"session-{uuid.uuid4()}"
    url = (f"{st.session_state.api_base}/apps/{st.session_state.app_name}"
           f"/users/{st.session_state.user_id}/sessions/{sid}")
    requests.post(url, json={}, headers={"Content-Type": "application/json"},
                  timeout=60).raise_for_status()
    st.session_state.session_id = sid
    return sid

def run_turn(question):
    return POST(f"{st.session_state.api_base}/run", {
        "app_name":   st.session_state.app_name,
        "user_id":    st.session_state.user_id,
        "session_id": st.session_state.session_id,
        "new_message": {"role": "user", "parts": [{"text": question}]},
    })

def normalize_events(resp):
    if isinstance(resp, list): return resp
    if isinstance(resp, dict) and isinstance(resp.get("events"), list):
        return resp["events"]
    return []

def last_text(events):
    text = ""
    for ev in events:
        for p in (ev.get("content") or {}).get("parts", []):
            if isinstance(p.get("text"), str) and p["text"].strip():
                text = p["text"].strip()
    return text

# ── Sidebar
with st.sidebar:
    st.subheader("Server & Session")
    st.text_input("ADK API Base", key="api_base")
    st.text_input("App Name",     key="app_name")
    st.text_input("User ID",      key="user_id")
    if st.button("Create / Reset Session", use_container_width=True):
        try:    st.success(f"Session: {create_session()}")
        except requests.HTTPError as e: st.error(str(e))
    if st.session_state.session_id:
        st.info(f"Active: {st.session_state.session_id}")
    else:
        st.warning("Create a session to begin.")

# ── Main
st.divider()
q   = st.text_input("Your question", "What is retrieval augmented generation?")
ask = st.button("Ask")
show_raw = st.checkbox("Show raw events", value=False)

if st.session_state.session_id and ask:
    try:
        resp   = run_turn(q)
        events = normalize_events(resp)
        st.session_state.events = events
        st.success("Answer:")
        st.write(last_text(events) or "_(No final text found)_")
    except requests.HTTPError as e:
        st.error(f"/run failed: {e}")

if show_raw and st.session_state.events:
    st.subheader("Raw events")
    st.code(json.dumps(st.session_state.events, indent=2), language="json")


Writing project_1/apps/app.py


### Step 6 — Live Demo via Python SDK (Colab path)
Same agent logic — no HTTP server needed.


In [30]:
import sys, importlib
sys.path.insert(0, 'project_1')
os.environ['ADK_MODEL'] = DEFAULT_MODEL

import agents.simple.agent as _simple_mod
importlib.reload(_simple_mod)
simple_agent = _simple_mod.root_agent

from google.genai import types as genai_types

P1_APP  = 'simple_qa_demo'
P1_USER = 'colab_qa_user'
P1_SID  = 'qa_session_01'
p1_service = InMemorySessionService()

async def run_qa_demo():
    await p1_service.create_session(app_name=P1_APP, user_id=P1_USER, session_id=P1_SID)
    runner = Runner(agent=simple_agent, app_name=P1_APP, session_service=p1_service)
    questions = [
        'What is retrieval augmented generation?',
        'Explain transformer attention in simple terms.',
        'Difference between fine-tuning and prompt engineering?',
    ]
    for q in questions:
        print(f'\nQ: {q}')
        content = genai_types.Content(role='user', parts=[genai_types.Part(text=q)])
        async for event in runner.run_async(user_id=P1_USER, session_id=P1_SID, new_message=content):
            if event.is_final_response() and event.content and event.content.parts:
                print(f"A: {(event.content.parts[0].text or '').strip()[:600]}")

asyncio.run(run_qa_demo())



Q: What is retrieval augmented generation?
A: Retrieval augmented generation is a technique used in natural language processing (NLP) and artificial intelligence (AI) that combines the strengths of retrieval-based and generation-based approaches. It involves retrieving relevant information from a database or knowledge base and then using this information to generate text or responses. This approach is often used in applications such as chatbots, language translation, and text summarization. According to research papers and NLP literature, retrieval augmented generation can improve the accuracy and fluency of generated text by leveraging t

Q: Explain transformer attention in simple terms.
A: Transformer attention is a mechanism used in neural networks to focus on specific parts of the input data when generating output. It's like a spotlight that highlights the most relevant information. Imagine you're trying to answer a question about a sentence, and you need to focus on certain words

### Step 7 — API Server + Streamlit reference (run outside Colab)

```bash
# Terminal 1 — launch the ADK API server
cd project_1
export ADK_MODEL=groq/llama-3.3-70b-versatile
export GROQ_API_KEY=your_key
adk api_server -v .
# → http://localhost:8000/docs  (Swagger UI)

# Terminal 2 — test with curl
APP=simple; USER=demo; SID=session-1
curl -s -X POST "http://localhost:8000/apps/$APP/users/$USER/sessions/$SID" \
     -H 'Content-Type: application/json' -d '{}'

curl -s -X POST http://localhost:8000/run \
     -H 'Content-Type: application/json' \
     -d '{"app_name":"simple","user_id":"demo","session_id":"session-1",
          "new_message":{"role":"user","parts":[{"text":"What is RAG?"}]}}'

# Terminal 3 — launch Streamlit
streamlit run project_1/apps/app.py
```


---
# ☁️ Project 2 — Capital Agent
### Minimal Agent Structure

**What you learn:**
- Smallest possible ADK agent (`root_agent`, no tools)
- Folder structure required by `adk deploy cloud_run`
- Deployment to Google Cloud Run (full walkthrough in the markdown section below)

**Folder structure (mirrors course):**
```
project_2/
└── capital_agent/
    ├── __init__.py
    ├── agent.py         ← root_agent
    └── requirements.txt
```


### Step 1 — Create folder structure

In [31]:
os.makedirs('project_2/capital_agent', exist_ok=True)
print('project_2/ structure ready')


project_2/ structure ready


In [32]:
%%writefile project_2/capital_agent/__init__.py
# Package marker


Writing project_2/capital_agent/__init__.py


In [33]:
%%writefile project_2/capital_agent/agent.py
"""
Capital Agent — minimal ADK agent (adapted for Groq/LLaMA).
Original: gemini-2.0-flash  |  Adapted: groq/llama-3.1-8b-instant
"""
import os
from google.adk.agents.llm_agent import LlmAgent

MODEL = os.getenv("ADK_MODEL", "groq/llama-3.1-8b-instant")

root_agent = LlmAgent(
    model=MODEL,
    name="capital_agent",
    instruction=(
        "You answer concisely. If asked for a country's capital, reply with just the capital name "
        "and a short confirmation (one sentence). For other questions, answer helpfully in 2-4 sentences."
    ),
)


Writing project_2/capital_agent/agent.py


In [34]:
%%writefile project_2/capital_agent/requirements.txt
# Agent dependencies for Cloud Run container
google-adk
python-dotenv
groq
litellm


Writing project_2/capital_agent/requirements.txt


### Step 3 — Live Demo

In [35]:
sys.path.insert(0, 'project_2')
os.environ['ADK_MODEL'] = INSTANT_MODEL

import capital_agent.agent as _cap_mod
importlib.reload(_cap_mod)
capital_root_agent = _cap_mod.root_agent

P2_APP  = 'capital_agent_app'
P2_USER = 'colab_capital_user'
P2_SID  = 'capital_session_01'
p2_service = InMemorySessionService()

async def run_capital_demo():
    await p2_service.create_session(app_name=P2_APP, user_id=P2_USER, session_id=P2_SID)
    runner = Runner(agent=capital_root_agent, app_name=P2_APP, session_service=p2_service)
    for q in [
        'What is the capital of France?',
        'What is the capital of Japan?',
        'What is the capital of Brazil?',
        'Tell me something interesting about world capitals.',
    ]:
        print(f'\nQ: {q}')
        content = genai_types.Content(role='user', parts=[genai_types.Part(text=q)])
        async for event in runner.run_async(user_id=P2_USER, session_id=P2_SID, new_message=content):
            if event.is_final_response() and event.content and event.content.parts:
                print(f"A: {(event.content.parts[0].text or '').strip()}")

asyncio.run(run_capital_demo())



Q: What is the capital of France?
A: Paris. It is the centre of French politics and culture.

Q: What is the capital of Japan?
A: Tokyo. It is a major economic and cultural hub in the region.

Q: What is the capital of Brazil?
A: Brasília. It was specifically designed to be the country's capital in the 20th century.

Q: Tell me something interesting about world capitals.
A: Many world capitals are intentionally built in a neutral or easily defensible location.


---
# ☁️ Cloud Run Deployment Walkthrough
## *(Documentation only — run these commands in a terminal, not in Colab)*

---

## Prerequisites

| Requirement | Details |
|---|---|
| Google Cloud project | Billing enabled |
| `gcloud` CLI | Authenticated via `gcloud auth login` |
| ADK CLI | `pip install google-adk` |
| APIs enabled | Cloud Run, Cloud Build, Artifact Registry |

```bash
# Enable required Google Cloud APIs
gcloud services enable run.googleapis.com \
                        cloudbuild.googleapis.com \
                        artifactregistry.googleapis.com
```

---

## Step 1 — Export environment variables

```bash
export GOOGLE_CLOUD_PROJECT='your-project-id'
export GOOGLE_CLOUD_LOCATION='us-central1'
export GROQ_API_KEY='your-groq-api-key'
```

---

## Step 2 — Store GROQ_API_KEY in Secret Manager (recommended)

```bash
echo -n "$GROQ_API_KEY" | gcloud secrets create groq-api-key --data-file=-
```

---

## Step 3 — Deploy with `adk deploy cloud_run`

Run from the **`project_2/`** directory:

```bash
cd project_2

adk deploy cloud_run \
  --project=$GOOGLE_CLOUD_PROJECT \
  --region=$GOOGLE_CLOUD_LOCATION \
  --service_name=capital-service \
  --app_name=capital-agent-app \
  --with_ui \
  capital_agent
```

**What this does:**
1. Packages `capital_agent/` into a Docker container via Cloud Build
2. Pushes image to Artifact Registry
3. Deploys to Cloud Run as `capital-service`
4. Exposes `/run`, `/list-apps`, and web UI endpoints

---

## Step 4 — Test the live service

```bash
SERVICE_URL=$(gcloud run services describe capital-service \
  --region=$GOOGLE_CLOUD_LOCATION --format='value(status.url)')

APP=capital-agent-app; USER=demo_user; SID="session-$(date +%s)"

# Create a session
curl -s -X POST "$SERVICE_URL/apps/$APP/users/$USER/sessions/$SID" \
     -H 'Content-Type: application/json' \
     -H "Authorization: Bearer $(gcloud auth print-identity-token)" \
     -d '{}'

# Run a turn
curl -s -X POST "$SERVICE_URL/run" \
     -H 'Content-Type: application/json' \
     -H "Authorization: Bearer $(gcloud auth print-identity-token)" \
     -d '{"app_name":"capital-agent-app","user_id":"demo_user","session_id":"'$SID'",
           "new_message":{"role":"user","parts":[{"text":"Capital of India?"}]}}'
```

---

## Step 5 — (Optional) Allow unauthenticated access

```bash
gcloud run services add-iam-policy-binding capital-service \
  --region=$GOOGLE_CLOUD_LOCATION \
  --member='allUsers' \
  --role='roles/run.invoker'
```

---

## Step 6 — Clean up

```bash
gcloud run services delete capital-service \
  --region=$GOOGLE_CLOUD_LOCATION --quiet
```

---

## Deployment architecture

```
Client (browser / curl)
        │
        ▼
Cloud Run Service  (capital-service)
  ┌─────────────────────────────────────────┐
  │  ADK API Server  (bundled by deployer)  │
  │   POST /run  ──►  capital_agent/agent   │
  │   GET  /list-apps                       │
  │   GET/POST /apps/.../sessions/...       │
  └─────────────────────────────────────────┘
        │
        ▼
  Groq API  (llama-3.1-8b-instant)
```


---
# ✅ What We Covered

| Project | Core ADK concepts | Groq model |
|---|---|---|
| 0 — Persistent Storage | `InMemorySessionService`, `DatabaseSessionService`, `LlmAgent` + tools, `ToolContext.state` CRUD | `llama-3.3-70b-versatile` |
| 1 — API Server + Streamlit | `Runner.run_async`, event streaming, `ADKClient`, Streamlit integration | `llama-3.3-70b-versatile` |
| 2 — Capital Agent + Cloud Run | Minimal `root_agent`, `adk deploy cloud_run`, Cloud Run testing | `llama-3.1-8b-instant` |

### Key pattern (all three projects)
```
User Input
  → Runner.run_async()
      → LlmAgent  →  Groq/LLaMA via LiteLLM bridge
          → (optional) Tool mutates tool_context.state
              → SessionService persists delta
                  → Final response streamed to caller
```

### Handy ADK CLI commands
```bash
adk web -v .               # Web UI for any project with root_agent
adk api_server -v .        # REST API server (port 8000)
adk deploy cloud_run ...   # Deploy to Google Cloud Run
```
